# Resource Scheduling with OR-Tools CP-SAT

This notebook demonstrates scheduling resources (people) to project activities using Google OR-Tools CP-SAT solver.

## Applicability

Out of box, OR-Tools is powerful and rich set of models and combination of CP (constraint programming) and SAT (boolean satisfaction) is impressive. The reason is that the DPLL algorithm, which powers SAT solvers is surprisingly fast. And SAT is a classical NP-complete problem. So if there's a way to reduce the problem to SAT we'll have potentially a feasable solution for some of the problems in the class. The key word is "potentially" as NP-complete problems means there's no known polynomial time solution, so for sufficiently large problem it's a game of chance to know if powerful heuristics fueling the SAT solver will deliver any solution at all within reasonable time. Still in practice unless we are dealing with billions of parameters we have a very good chance to find something useful.

## The Challenge

CP SAT is

a) low level
b) academically focused

In this notebook there's an attempt to make it a bit more structured and implement basic soft constraint approach via optimisations


## Approach

We use CP-SAT with an annotation-based constraint system:
- **Hard constraints**: Enforced as CP-SAT constraints
- **Soft constraints**: Combined into a minimization objective function

The constraint provider class uses annotations to distinguish between hard and soft constraints, and automatically builds the optimization model.

## CP-SAT Domain Objects
This implementation uses OR-Tools's `Domain` class to represent valid integer ranges:
- **Pre-computed domains**: Calculated once during problem setup, reused during solving
- **Efficient propagation**: CP-SAT can propagate domain restrictions more effectively than individual constraints
- **Reduced constraint count**: One domain constraint replaces many equality/inequality constraints

### Domain Examples
- `weekdayDatesDomain`: Pre-filtered to exclude weekends
- `workingDateDomainByResource[R]`: Valid dates for resource R (no weekends/bank holidays)
- `getResourceDomainBySkill(S)`: Resources filtered by skill level S
- `getDateDomainForProject(P)`: Dates within project P's boundaries


## Setup boilerplate


In [1]:
// OR-Tools dependencies
@file:DependsOn("../deps/protobuf-java-4.31.1.jar")  
@file:DependsOn("../deps/ortools-java-9.14.6206.jar")
@file:DependsOn("../deps/ortools-darwin-aarch64-9.14.6206.jar")
@file:DependsOn("../deps/ortools-darwin-x86-64-9.14.6206.jar")
@file:DependsOn("../deps/jna-5.14.0.jar")
@file:DependsOn("../deps/jna-platform-5.14.0.jar")

// Jackson dependencies for YAML parsing
@file:DependsOn("../deps/jackson-core-2.18.0.jar")
@file:DependsOn("../deps/jackson-databind-2.18.0.jar")
@file:DependsOn("../deps/jackson-annotations-2.18.0.jar")
@file:DependsOn("../deps/jackson-dataformat-yaml-2.18.0.jar")
@file:DependsOn("../deps/jackson-datatype-jsr310-2.18.0.jar")
@file:DependsOn("../deps/jackson-module-kotlin-2.18.0.jar")
@file:DependsOn("../deps/snakeyaml-2.3.jar")

// Other dependencies
@file:DependsOn("../deps/colormath-3.5.0.jar")


In [2]:
// Import OR-Tools classes
import com.google.ortools.Loader
import com.google.ortools.sat.*
import com.google.ortools.util.Domain
import com.google.ortools.sat.LinearExpr
import com.google.ortools.sat.LinearExprBuilder
import com.google.ortools.sat.CpModel
import com.google.ortools.sat.IntVar
import com.google.ortools.sat.Literal

// Import other dependencies
import java.time.LocalDate
import java.time.DayOfWeek
import com.fasterxml.jackson.databind.ObjectMapper
import com.fasterxml.jackson.databind.module.SimpleModule
import com.fasterxml.jackson.databind.JsonDeserializer
import com.fasterxml.jackson.databind.DeserializationContext
import com.fasterxml.jackson.core.JsonParser
import com.fasterxml.jackson.core.type.TypeReference
import com.fasterxml.jackson.dataformat.yaml.YAMLFactory
import com.fasterxml.jackson.datatype.jsr310.JavaTimeModule
import com.fasterxml.jackson.module.kotlin.KotlinModule
import com.fasterxml.jackson.annotation.JsonProperty
import java.io.File
import com.github.ajalt.colormath.model.RGB
import com.github.ajalt.colormath.model.HSL
import kotlin.random.Random
import kotlin.reflect.KClass
import kotlin.reflect.full.*

// Load OR-Tools native libraries
com.google.ortools.Loader.loadNativeLibraries()


## Annotations

Define annotations to mark methods as hard or soft constraints as we did in timefold example.


In [3]:
@Target(AnnotationTarget.FUNCTION)
@Retention(AnnotationRetention.RUNTIME)
annotation class HardConstraint(val name: String)

@Target(AnnotationTarget.FUNCTION)
@Retention(AnnotationRetention.RUNTIME)
annotation class SoftConstraint(val name: String, val weight: Int = 1)


## Simple Relational Model

Same data model as the Timefold version, using type aliases for domain IDs.


In [4]:
// Type aliases for domain model IDs
typealias ProjectID = Long
typealias ResourceID = Long
typealias ActivityID = Long
typealias DepartmentID = String
typealias PositionID = String
typealias SkillLevelID = String
typealias BankHolidaySetID = String


## Data Classes


In [ ]:
data class Department(
    @JsonProperty("id")
    val id: DepartmentID,

    @JsonProperty("name")
    val name: String
)

data class Position(
    @JsonProperty("id")
    val id: PositionID,

    @JsonProperty("name")
    val name: String,

    @JsonProperty("skillLevel")
    val skillLevel: SkillLevelID,

    @JsonProperty("hourlyRate")
    val hourlyRate: Double
)

data class BankHolidaySet(
    @JsonProperty("id")
    val id: BankHolidaySetID,

    @JsonProperty("name")
    val name: String,

    @JsonProperty("holidays")
    val holidays: List<LocalDate> = emptyList()
)

data class Resource(
    @JsonProperty("id")
    val id: ResourceID,

    @JsonProperty("name")
    val name: String,

    @JsonProperty("departmentId")
    val departmentId: DepartmentID,

    @JsonProperty("positionId")
    val positionId: PositionID,

    @JsonProperty("skills")
    val skills: List<SkillLevelID> = emptyList(),

    @JsonProperty("bankHolidaySetId")
    val bankHolidaySetId: BankHolidaySetID,

    @JsonProperty("weeklyAvailability")
    val weeklyAvailability: Map<String, Int> = emptyMap()
)

data class Project(
    @JsonProperty("id")
    val id: ProjectID,

    @JsonProperty("name")
    val name: String,

    @JsonProperty("projectCode")
    val projectCode: String,

    @JsonProperty("departmentId")
    val departmentId: DepartmentID,

    @JsonProperty("startDate")
    val startDate: LocalDate?,

    @JsonProperty("endDate")
    val endDate: LocalDate?,

    @JsonProperty("priority")
    val priority: Int
)

data class Activity(
    @JsonProperty("id")
    val id: ActivityID,

    @JsonProperty("code")
    val code: String,

    @JsonProperty("name")
    val name: String,

    @JsonProperty("order")
    val order: Int,

    @JsonProperty("projectId")
    val projectId: ProjectID,

    @JsonProperty("estimatedHours")
    val estimatedHours: Int,

    @JsonProperty("rate")
    val rate: Double,

    @JsonProperty("isCommercialRate")
    val isCommercialRate: Boolean,

    @JsonProperty("budgetNumbers")
    val budgetNumbers: Int? = null,

    @JsonProperty("requestDepartmentId")
    val requestDepartmentId: DepartmentID? = null,

    @JsonProperty("requestPositionId")
    val requestPositionId: PositionID? = null,

    @JsonProperty("requestSkillLevel1")
    val requestSkillLevel1: SkillLevelID? = null,

    @JsonProperty("requestSkillLevel2")
    val requestSkillLevel2: SkillLevelID? = null,

    @JsonProperty("requestSkillLevel3")
    val requestSkillLevel3: SkillLevelID? = null
) {
    fun calculateMargin(resourceHourlyRate: Double): Double {
        return if (isCommercialRate) rate - resourceHourlyRate else 0.0
    }
}

data class DataModel(
    @JsonProperty("departments")
    val departments: List<Department> = emptyList(),

    @JsonProperty("positions")
    val positions: List<Position> = emptyList(),

    @JsonProperty("bankHolidaySets")
    val bankHolidaySets: List<BankHolidaySet> = emptyList(),

    @JsonProperty("resources")
    val resources: List<Resource> = emptyList(),

    @JsonProperty("projects")
    val projects: List<Project> = emptyList(),

    @JsonProperty("activities")
    val activities: List<Activity> = emptyList()
)


## Problem class

As in timefold we need some variable which model is allowed to change, basically we play a game of assigning activityIds and projectIds to workunits and then model timewax using this hour-long assignments using combination of hard and soft constraints.


In [6]:
// Work unit - represents 1 hour of work on an activity
// The solver assigns each hour to a resource and date
// Bookings are created by grouping units by (activity, resource, date)
data class WorkUnit(
    val id: Long,
    val activityId: ActivityID,
    val projectId: ProjectID
)

// Problem data container that stores all problem data and pre-computes CP-SAT Domain objects
data class ResourceScheduleProblem(
    val projects: List<Project>,
    val resources: List<Resource>,
    val activities: List<Activity>,
    val departments: List<Department>,
    val positions: List<Position>,
    val bankHolidaySets: List<BankHolidaySet>,
    val dateRange: List<LocalDate>,
    val workUnits: List<WorkUnit>
) {
    // ============ ORDERED COLLECTIONS FOR PERFORMANCE ============
    // Store data in ordered collections indexed by their position
    private val resourcesIndexed: List<Resource> = resources.sortedBy { it.id }
    private val activitiesIndexed: List<Activity> = activities.sortedBy { it.id }
    private val projectsIndexed: List<Project> = projects.sortedBy { it.id }
    private val workUnitsIndexed: List<WorkUnit> = workUnits.sortedBy { it.id }
    private val datesIndexed: List<LocalDate> = dateRange.sorted()

    // ============ ID TO INDEX MAPPINGS (for CP-SAT integer domains) ============
    // Map domain IDs to integer indices [0, n-1]
    private val resourceIdToIndex: Map<ResourceID, Long> = resourcesIndexed.mapIndexed { idx, res -> res.id to idx.toLong() }.toMap()
    private val activityIdToIndex: Map<ActivityID, Long> = activitiesIndexed.mapIndexed { idx, act -> act.id to idx.toLong() }.toMap()
    private val projectIdToIndex: Map<ProjectID, Long> = projectsIndexed.mapIndexed { idx, proj -> proj.id to idx.toLong() }.toMap()
    private val workUnitIdToIndex: Map<Long, Long> = workUnitsIndexed.mapIndexed { idx, unit -> unit.id to idx.toLong() }.toMap()
    private val dateToIndex: Map<LocalDate, Long> = datesIndexed.mapIndexed { idx, date -> date to idx.toLong() }.toMap()

    // ============ INDEX TO ID MAPPINGS (reverse lookups) ============
    // Map integer indices back to domain IDs
    private val indexToResourceId: Map<Long, ResourceID> = resourcesIndexed.mapIndexed { idx, res -> idx.toLong() to res.id }.toMap()
    private val indexToActivityId: Map<Long, ActivityID> = activitiesIndexed.mapIndexed { idx, act -> idx.toLong() to act.id }.toMap()
    private val indexToProjectId: Map<Long, ProjectID> = projectsIndexed.mapIndexed { idx, proj -> idx.toLong() to proj.id }.toMap()
    private val indexToWorkUnitId: Map<Long, Long> = workUnitsIndexed.mapIndexed { idx, unit -> idx.toLong() to unit.id }.toMap()
    private val indexToDate: Map<Long, LocalDate> = datesIndexed.mapIndexed { idx, date -> idx.toLong() to date }.toMap()

    // ============ FAST LOOKUP MAPS ============
    // O(1) lookups by ID
    private val resourceById: Map<ResourceID, Resource> = resourcesIndexed.associateBy { it.id }
    private val activityById: Map<ActivityID, Activity> = activitiesIndexed.associateBy { it.id }
    private val projectById: Map<ProjectID, Project> = projectsIndexed.associateBy { it.id }
    private val workUnitById: Map<Long, WorkUnit> = workUnitsIndexed.associateBy { it.id }
    private val positionById: Map<PositionID, Position> = positions.associateBy { it.id }
    private val departmentById: Map<DepartmentID, Department> = departments.associateBy { it.id }
    private val bankHolidaySetById: Map<BankHolidaySetID, BankHolidaySet> = bankHolidaySets.associateBy { it.id }

    // ============ ID TO INDEX CONVERSION METHODS ============
    fun getResourceIndex(id: ResourceID): Long = resourceIdToIndex[id] ?: throw IllegalArgumentException("Resource ID $id not found")
    fun getActivityIndex(id: ActivityID): Long = activityIdToIndex[id] ?: throw IllegalArgumentException("Activity ID $id not found")
    fun getProjectIndex(id: ProjectID): Long = projectIdToIndex[id] ?: throw IllegalArgumentException("Project ID $id not found")
    fun getWorkUnitIndex(id: Long): Long = workUnitIdToIndex[id] ?: throw IllegalArgumentException("WorkUnit ID $id not found")
    fun getDateIndex(date: LocalDate): Long = dateToIndex[date] ?: throw IllegalArgumentException("Date $date not found")

    // ============ INDEX TO ID CONVERSION METHODS ============
    fun getResourceIdByIndex(idx: Long): ResourceID = indexToResourceId[idx] ?: throw IllegalArgumentException("Resource index $idx not found")
    fun getActivityIdByIndex(idx: Long): ActivityID = indexToActivityId[idx] ?: throw IllegalArgumentException("Activity index $idx not found")
    fun getProjectIdByIndex(idx: Long): ProjectID = indexToProjectId[idx] ?: throw IllegalArgumentException("Project index $idx not found")
    fun getWorkUnitIdByIndex(idx: Long): Long = indexToWorkUnitId[idx] ?: throw IllegalArgumentException("WorkUnit index $idx not found")
    fun getDateByIndex(idx: Long): LocalDate = indexToDate[idx] ?: throw IllegalArgumentException("Date index $idx not found")

    // ============ DATA ACCESS BY ID (O(1)) ============
    fun getResource(id: ResourceID): Resource? = resourceById[id]
    fun getActivity(id: ActivityID): Activity? = activityById[id]
    fun getProject(id: ProjectID): Project? = projectById[id]
    fun getWorkUnit(id: Long): WorkUnit? = workUnitById[id]
    fun getPosition(id: PositionID): Position? = positionById[id]
    fun getDepartment(id: DepartmentID): Department? = departmentById[id]
    fun getBankHolidaySet(id: BankHolidaySetID): BankHolidaySet? = bankHolidaySetById[id]

    // ============ DATA ACCESS BY INDEX (O(1)) ============
    fun getResourceByIndex(idx: Long): Resource = resourcesIndexed[idx.toInt()]
    fun getActivityByIndex(idx: Long): Activity = activitiesIndexed[idx.toInt()]
    fun getProjectByIndex(idx: Long): Project = projectsIndexed[idx.toInt()]
    fun getWorkUnitByIndex(idx: Long): WorkUnit = workUnitsIndexed[idx.toInt()]

    // ============ DOMAIN SIZE METHODS (for CP-SAT variable bounds) ============
    fun getResourceDomainSize(): Long = resourcesIndexed.size.toLong()
    fun getActivityDomainSize(): Long = activitiesIndexed.size.toLong()
    fun getProjectDomainSize(): Long = projectsIndexed.size.toLong()
    fun getWorkUnitDomainSize(): Long = workUnitsIndexed.size.toLong()
    fun getDateDomainSize(): Long = datesIndexed.size.toLong()

    // ============ INDEXED COLLECTION ACCESS ============
    fun getAllResourcesIndexed(): List<Resource> = resourcesIndexed
    fun getAllActivitiesIndexed(): List<Activity> = activitiesIndexed
    fun getAllProjectsIndexed(): List<Project> = projectsIndexed
    fun getAllWorkUnitsIndexed(): List<WorkUnit> = workUnitsIndexed
    fun getAllDatesIndexed(): List<LocalDate> = datesIndexed

    // ============ CP-SAT DOMAIN OBJECTS ============

    // Full domains for all entities
    val allResourcesDomain: Domain = Domain.fromValues(LongArray(resourcesIndexed.size) { it.toLong() })
    val allDatesDomain: Domain = Domain.fromValues(LongArray(datesIndexed.size) { it.toLong() })

    // Weekday dates domain (excludes weekends)
    private val _weekdayDatesDomain by lazy {
        val weekdayIndices = datesIndexed
            .mapIndexed { idx, date -> idx to date }
            .filter { (_, date) -> date.dayOfWeek != DayOfWeek.SATURDAY && date.dayOfWeek != DayOfWeek.SUNDAY }
            .map { it.first.toLong() }
        Domain.fromValues(weekdayIndices.toLongArray())
    }
    
    val weekdayDatesDomain: Domain
        get() = _weekdayDatesDomain

    // Date indices by their index for quick lookup
    private val weekendDateIndices by lazy {
        datesIndexed
            .mapIndexed { idx, date -> idx to date }
            .filter { (_, date) -> date.dayOfWeek == DayOfWeek.SATURDAY || date.dayOfWeek == DayOfWeek.SUNDAY }
            .map { it.first }
            .toSet()
    }

    // Resources grouped by department for fast domain creation
    private val resourcesByDepartment by lazy {
        resourcesIndexed
            .mapIndexed { idx, resource -> idx to resource }
            .groupBy { it.second.departmentId }
            .mapValues { (_, pairs) -> pairs.map { it.first } }
    }

    // Resources grouped by skill level for fast domain creation
    private val resourcesBySkill by lazy {
        resourcesIndexed
            .flatMapIndexed { idx, resource ->
                resource.skills.map { skill -> skill to idx }
            }
            .groupBy { it.first }
            .mapValues { (_, pairs) -> pairs.map { it.second }.distinct() }
    }

    // Bank holiday date indices by resource
    private val bankHolidayDateIndicesByResource by lazy {
        resourcesIndexed.associate { resource ->
            val bankHolidaySet = getBankHolidaySet(resource.bankHolidaySetId)
            val holidayIndices = bankHolidaySet?.holidays?.mapNotNull { holiday ->
                dateToIndex[holiday]
            }?.toSet() ?: emptySet<Long>()
            resource.id to holidayIndices
        }
    }

    // Working date domains by resource (excludes weekends and bank holidays)
    private val workingDateDomainByResource by lazy {
        resourcesIndexed.associate { resource ->
            val bankHolidayIndices = bankHolidayDateIndicesByResource[resource.id] ?: emptySet<Long>()
            val validIndices = datesIndexed
                .mapIndexed { idx, date -> idx to date }
                .filter { (idx, date) ->
                    date.dayOfWeek != DayOfWeek.SATURDAY &&
                    date.dayOfWeek != DayOfWeek.SUNDAY &&
                    idx.toLong() !in bankHolidayIndices
                }
                .map { it.first.toLong() }
            resource.id to Domain.fromValues(validIndices.toLongArray())
        }
    }

    // Date domains by project (within project boundaries)
    private val dateDomainByProject by lazy {
        projectsIndexed.associate { project ->
            val validIndices = datesIndexed
                .mapIndexed { idx, date -> idx to date }
                .filter { (_, date) ->
                    (project.startDate == null || !date.isBefore(project.startDate)) &&
                    (project.endDate == null || !date.isAfter(project.endDate))
                }
                .map { it.first.toLong() }
            project.id to Domain.fromValues(validIndices.toLongArray())
        }
    }

    // ============ CP-SAT DOMAIN ACCESS METHODS ============

    // Get resource domain filtered by department
    fun getResourceDomainByDepartment(departmentId: DepartmentID): Domain {
        val indices = resourcesByDepartment[departmentId]?.map { it.toLong() } ?: emptyList<Long>()
        return if (indices.isEmpty()) Domain.fromValues(longArrayOf())
               else Domain.fromValues(indices.toLongArray())
    }

    // Get resource domain filtered by skill level
    fun getResourceDomainBySkill(skillLevel: SkillLevelID): Domain {
        val indices = resourcesBySkill[skillLevel]?.map { it.toLong() } ?: emptyList<Long>()
        return if (indices.isEmpty()) Domain.fromValues(longArrayOf())
               else Domain.fromValues(indices.toLongArray())
    }

    // Get resource domain filtered by multiple skills (intersection)
    fun getResourceDomainBySkills(skillLevels: List<SkillLevelID>): Domain {
        if (skillLevels.isEmpty()) return allResourcesDomain

        val resourceSets = skillLevels.mapNotNull { resourcesBySkill[it]?.toSet() }
        if (resourceSets.isEmpty()) return Domain.fromValues(longArrayOf())

        val intersection = resourceSets.reduce { acc, set -> acc.intersect(set) }
        val indices = intersection.sorted().map { it.toLong() }
        return if (indices.isEmpty()) Domain.fromValues(longArrayOf())
               else Domain.fromValues(indices.toLongArray())
    }

    // Get resource domain filtered by department AND skill
    fun getResourceDomainByDepartmentAndSkill(departmentId: DepartmentID, skillLevel: SkillLevelID): Domain {
        val deptResources = resourcesByDepartment[departmentId]?.toSet() ?: emptySet<Int>()
        val skillResources = resourcesBySkill[skillLevel]?.toSet() ?: emptySet<Int>()
        val intersection = deptResources.intersect(skillResources)
        val indices = intersection.sorted().map { it.toLong() }
        return if (indices.isEmpty()) Domain.fromValues(longArrayOf())
               else Domain.fromValues(indices.toLongArray())
    }

    // Get working date domain for a specific resource (no weekends, no bank holidays)
    fun getWorkingDateDomainForResource(resourceId: ResourceID): Domain {
        return workingDateDomainByResource[resourceId] ?: weekdayDatesDomain
    }

    // Get date domain for a specific project (within project boundaries)
    fun getDateDomainForProject(projectId: ProjectID): Domain {
        return dateDomainByProject[projectId] ?: allDatesDomain
    }

    // Get date domain for a project AND resource (project boundaries + no weekends/holidays)
    fun getDateDomainForProjectAndResource(projectId: ProjectID, resourceId: ResourceID): Domain {
        val projectDates = dateDomainByProject[projectId] ?: allDatesDomain
        val resourceDates = workingDateDomainByResource[resourceId] ?: weekdayDatesDomain

        // Intersect the two domains
        // flattenedIntervals() returns long[] where pairs [min, max, min, max, ...] represent intervals
        val projectIntervals = projectDates.flattenedIntervals()
        val projectDateSet = mutableSetOf<Int>()
        var i = 0
        while (i < projectIntervals.size) {
            val min = projectIntervals[i].toInt()
            val max = projectIntervals[i + 1].toInt()
            for (value in min..max) {
                projectDateSet.add(value)
            }
            i += 2
        }

        val resourceIntervals = resourceDates.flattenedIntervals()
        val resourceDateSet = mutableSetOf<Int>()
        i = 0
        while (i < resourceIntervals.size) {
            val min = resourceIntervals[i].toInt()
            val max = resourceIntervals[i + 1].toInt()
            for (value in min..max) {
                resourceDateSet.add(value)
            }
            i += 2
        }

        val intersection = projectDateSet.intersect(resourceDateSet).sorted().map { it.toLong() }
        return if (intersection.isEmpty()) Domain.fromValues(longArrayOf())
               else Domain.fromValues(intersection.toLongArray())
    }

    // Check if a date index is a weekend
    fun isWeekendDate(dateIdx: Int): Boolean = dateIdx in weekendDateIndices

    // Check if a date index is a bank holiday for a resource
    fun isBankHolidayForResource(dateIdx: Int, resourceId: ResourceID): Boolean {
        return bankHolidayDateIndicesByResource[resourceId]?.contains(dateIdx.toLong()) ?: false
    }
}


## Solution Container


In [7]:
// Solution container class that holds the problem, assignments and metrics
data class ResourceScheduleSolution(
    val problem: ResourceScheduleProblem,
    val assignments: Map<Long, Pair<ResourceID, LocalDate>>,  // workUnitId -> (resourceId, date)
    val objectiveValue: Long,
    val solveTimeMs: Long
)


## CP-SAT Model 

We separate the model definition from the solver implementation:
- **Model Interface**: Defines the contract for constraint models
- **Scheduling Model**: Contains problem facts, variables, and constraint definitions with annotations
- **CP-SAT Builder**: Adapts the model to CP-SAT and solves it

This separation allows the model to be solver-agnostic.


In [8]:
// ============ INTERFACE ============

interface ConstraintModel {
    val problem: ResourceScheduleProblem
}

// ============ MODEL CLASS ============

/**
 * Scheduling constraint model using CP-SAT Domain objects for efficient constraint posting.
 *
 * This model leverages pre-computed Domain objects from the problem container to:
 * - Avoid recomputing valid value ranges during constraint posting
 * - Reduce number of variables and constraints needed
 * - Allow CP-SAT to efficiently propagate domain restrictions
 *
 * Key efficiency gains:
 * - Weekend/holiday restrictions use pre-computed date domains
 * - Skill/department filters use pre-computed resource domains
 * - Project boundaries use pre-computed date domains
 * - Domain intersections are computed once and reused
 */
class SchedulingModel(
    override val problem: ResourceScheduleProblem
) : ConstraintModel {

    // Variable definitions (accessed by builder)
    lateinit var resourceVars: Map<Long, IntVar>
    lateinit var dateVars: Map<Long, IntVar>
    lateinit var cpModel: CpModel

    // Soft penalties (collected by builder)
    val softPenalties = mutableListOf<LinearExpr>()

    // Helper methods for linear expressions
    private fun buildLinearExpr(): LinearExprBuilder = LinearExpr.newBuilder()

    private fun buildResourceExpr(resourceVar: IntVar, resourceIdx: Int): LinearExpr =
        buildLinearExpr()
            .add(resourceVar)
            .add(LinearExpr.constant(-resourceIdx.toLong()))
            .build()

    private fun buildDateExpr(dateVar: IntVar, dateIdx: Int): LinearExpr =
        buildLinearExpr()
            .add(dateVar)
            .add(LinearExpr.constant(-dateIdx.toLong()))
            .build()

    private fun buildResourceExprLong(resourceVar: IntVar, resourceIdx: Long): LinearExpr =
        buildLinearExpr()
            .add(resourceVar)
            .add(LinearExpr.constant(-resourceIdx))
            .build()

    private fun buildDateExprLong(dateVar: IntVar, dateIdx: Long): LinearExpr =
        buildLinearExpr()
            .add(dateVar)
            .add(LinearExpr.constant(-dateIdx))
            .build()

    // ============ HARD CONSTRAINTS ============

    @HardConstraint("Resource daily capacity (max 8 hours)")
    fun resourceDailyCapacity() {
        // For each resource and each date, count work units and enforce capacity
        // Uses CP-SAT's channeling features for efficient boolean reification
        for (resourceIdx in 0L until problem.getResourceDomainSize()) {
            val resourceId = problem.getResourceIdByIndex(resourceIdx)
            val resource = problem.getResourceByIndex(resourceIdx)
            
            for (dateIdx in 0L until problem.getDateDomainSize()) {
                val date = problem.getDateByIndex(dateIdx)
                
                // Get capacity for this resource on this day from weeklyAvailability
                val dayOfWeek = date.dayOfWeek.toString()
                val capacity = resource.weeklyAvailability[dayOfWeek] ?: 8
                
                // Skip if capacity is 0 (resource not available)
                if (capacity == 0) continue
                
                // Create boolean variables for each work unit: is it assigned to this resource on this date?
                val assignmentVars = problem.getAllWorkUnitsIndexed().map { unit ->
                    // Create intermediate boolean variables for each condition
                    val isResourceMatch = cpModel.newBoolVar("res_match_${unit.id}_r${resourceId}_d${dateIdx}")
                    val isDateMatch = cpModel.newBoolVar("date_match_${unit.id}_r${resourceId}_d${dateIdx}")
                    val isAssignedHere = cpModel.newBoolVar("assigned_${unit.id}_r${resourceId}_d${dateIdx}")
                    
                    // Channel: isResourceMatch <=> (resourceVar == resourceIdx)
                    // Forward: isResourceMatch => resourceVar == resourceIdx
                    cpModel.addEquality(buildResourceExprLong(resourceVars[unit.id]!!, resourceIdx), 0L)
                        .onlyEnforceIf(isResourceMatch)
                    // Backward: resourceVar != resourceIdx => !isResourceMatch
                    cpModel.addDifferent(buildResourceExprLong(resourceVars[unit.id]!!, resourceIdx), 0L)
                        .onlyEnforceIf(isResourceMatch.not())
                    
                    // Channel: isDateMatch <=> (dateVar == dateIdx)
                    // Forward: isDateMatch => dateVar == dateIdx
                    cpModel.addEquality(buildDateExprLong(dateVars[unit.id]!!, dateIdx), 0L)
                        .onlyEnforceIf(isDateMatch)
                    // Backward: dateVar != dateIdx => !isDateMatch
                    cpModel.addDifferent(buildDateExprLong(dateVars[unit.id]!!, dateIdx), 0L)
                        .onlyEnforceIf(isDateMatch.not())
                    
                    // Channel: isAssignedHere <=> (isResourceMatch AND isDateMatch)
                    // This uses CP-SAT's addBoolAnd for efficient conjunction
                    cpModel.addBoolAnd(arrayOf(isResourceMatch, isDateMatch)).onlyEnforceIf(isAssignedHere)
                    cpModel.addBoolOr(arrayOf(isResourceMatch.not(), isDateMatch.not())).onlyEnforceIf(isAssignedHere.not())
                    
                    isAssignedHere
                }
                
                // Sum of all work units assigned to this resource on this date must not exceed capacity
                // This is a linear constraint that CP-SAT handles very efficiently
                cpModel.addLessOrEqual(LinearExpr.sum(assignmentVars.toTypedArray()), capacity.toLong())
            }
        }
    }

    @HardConstraint("No weekend bookings")
    fun noWeekendBookings() {
        // Use pre-computed weekday domain for all work units
        for (unit in problem.getAllWorkUnitsIndexed()) {
            cpModel.addLinearExpressionInDomain(dateVars[unit.id]!!, problem.weekdayDatesDomain)
        }
    }

    @HardConstraint("No bank holiday bookings")
    fun noBankHolidayBookings() {
        for (unit in problem.getAllWorkUnitsIndexed()) {
            // For each resource, enforce working dates domain when assigned
            for (resourceIdx in 0L until problem.getResourceDomainSize()) {
                val resourceId = problem.getResourceIdByIndex(resourceIdx)
                val workingDatesDomain = problem.getWorkingDateDomainForResource(resourceId)
                
                // Create indicator for resource assignment
                val isAssigned = cpModel.newBoolVar("res_assigned_${unit.id}_${resourceId}")
                cpModel.addEquality(buildResourceExprLong(resourceVars[unit.id]!!, resourceIdx), 0L)
                    .onlyEnforceIf(isAssigned)
                
                // When assigned to this resource, date must be in working dates domain
                cpModel.addLinearExpressionInDomain(dateVars[unit.id]!!, workingDatesDomain)
                    .onlyEnforceIf(isAssigned)
            }
        }
    }

    @HardConstraint("Required skill level")
    fun requiredSkillLevel() {
        for (unit in problem.getAllWorkUnitsIndexed()) {
            val activity = problem.getActivity(unit.activityId)!!
            if (activity.requestSkillLevel1 != null) {
                // Get domain of resources with required skill
                val validResourceDomain = problem.getResourceDomainBySkill(activity.requestSkillLevel1)
                
                // Resource assignment must be in valid domain
                cpModel.addLinearExpressionInDomain(resourceVars[unit.id]!!, validResourceDomain)
            }
        }
    }

    @HardConstraint("Project date boundaries")
    fun projectDateBoundaries() {
        for (unit in problem.getAllWorkUnitsIndexed()) {
            // Use pre-computed date domain for the project
            val dateDomain = problem.getDateDomainForProject(unit.projectId)
            cpModel.addLinearExpressionInDomain(dateVars[unit.id]!!, dateDomain)
        }
    }

    @HardConstraint("Resource department match")
    fun resourceDepartmentMatch() {
        for (unit in problem.getAllWorkUnitsIndexed()) {
            val activity = problem.getActivity(unit.activityId)!!
            if (activity.requestDepartmentId != null) {
                // Get domain of resources in required department
                val validResourceDomain = problem.getResourceDomainByDepartment(activity.requestDepartmentId)
                
                // Resource assignment must be in valid domain
                cpModel.addLinearExpressionInDomain(resourceVars[unit.id]!!, validResourceDomain)
            }
        }
    }

    @HardConstraint("Activities must be completed in order")
    fun activitiesInOrder() {
        // Group work units by project
        val unitsByProject = problem.getAllWorkUnitsIndexed()
            .groupBy { it.projectId }
        
        for ((projectId, projectUnits) in unitsByProject) {
            // Get activities for this project in order
            val orderedActivities = problem.getAllActivitiesIndexed()
                .filter { it.projectId == projectId }
                .sortedBy { it.order }
            
            // For each pair of consecutive activities
            for (i in 0 until orderedActivities.size - 1) {
                val earlierActivity = orderedActivities[i]
                val laterActivity = orderedActivities[i + 1]
                
                // Get work units for both activities
                val earlierUnits = projectUnits.filter { it.activityId == earlierActivity.id }
                val laterUnits = projectUnits.filter { it.activityId == laterActivity.id }
                
                // Create variables for max date of earlier activity and min date of later activity
                val maxEarlierDate = cpModel.newIntVar(0, problem.getDateDomainSize() - 1, 
                    "max_date_${earlierActivity.id}")
                val minLaterDate = cpModel.newIntVar(0, problem.getDateDomainSize() - 1, 
                    "min_date_${laterActivity.id}")
                
                // Max date of earlier activity
                for (unit in earlierUnits) {
                    cpModel.addLessOrEqual(dateVars[unit.id]!!, maxEarlierDate)
                }
                
                // Min date of later activity
                for (unit in laterUnits) {
                    cpModel.addGreaterOrEqual(dateVars[unit.id]!!, minLaterDate)
                }
                
                // Ensure max date of earlier < min date of later
                cpModel.addLessThan(maxEarlierDate, minLaterDate)
            }
        }
    }

    // ============ SOFT CONSTRAINTS ============

    @SoftConstraint("Prefer same resource per activity", weight = 10)
    fun preferSameResourcePerActivity() {
        val unitsByActivity = problem.getAllWorkUnitsIndexed().groupBy { it.activityId }

        for ((_, units) in unitsByActivity) {
            if (units.size > 1) {
                for (i in 0 until units.size - 1) {
                    for (j in i + 1 until units.size) {
                        val unit1 = units[i]
                        val unit2 = units[j]

                        // Create a penalty expression that will be 1 if resources are different
                        val areEqual = cpModel.newBoolVar("same_res_${unit1.id}_${unit2.id}")
                        
                        // Link resources being equal to the boolean
                        cpModel.addEquality(
                            buildLinearExpr()
                                .add(resourceVars[unit1.id]!!)
                                .add(LinearExpr.term(resourceVars[unit2.id]!!, -1L))
                                .build(),
                            0L
                        ).onlyEnforceIf(areEqual)
                        
                        // Add penalty to objective (10 points per different resource)
                        softPenalties.add(
                            buildLinearExpr()
                                .add(LinearExpr.term(areEqual.not(), 10L))
                                .build()
                        )
                    }
                }
            }
        }
    }

    @SoftConstraint("Prefer consecutive days", weight = 5)
    fun preferConsecutiveDays() {
        val unitsByActivity = problem.getAllWorkUnitsIndexed().groupBy { it.activityId }

        for ((_, units) in unitsByActivity) {
            if (units.size > 1) {
                for (i in 0 until units.size - 1) {
                    for (j in i + 1 until units.size) {
                        val unit1 = units[i]
                        val unit2 = units[j]

                        // Create booleans for conditions
                        val sameResource = cpModel.newBoolVar("same_res_${unit1.id}_${unit2.id}")
                        val consecutive = cpModel.newBoolVar("consec_${unit1.id}_${unit2.id}")
                        
                        // Check resource equality
                        cpModel.addEquality(
                            buildLinearExpr()
                                .add(resourceVars[unit1.id]!!)
                                .add(LinearExpr.term(resourceVars[unit2.id]!!, -1L))
                                .build(),
                            0L
                        ).onlyEnforceIf(sameResource)
                        
                        // Check consecutive days
                        cpModel.addEquality(
                            buildLinearExpr()
                                .add(dateVars[unit2.id]!!)
                                .add(LinearExpr.term(dateVars[unit1.id]!!, -1L))
                                .build(),
                            1L
                        ).onlyEnforceIf(consecutive)
                        
                        // Both conditions must be true for reward
                        val bothTrue = cpModel.newBoolVar("both_${unit1.id}_${unit2.id}")
                        cpModel.addBoolAnd(arrayOf(sameResource, consecutive)).onlyEnforceIf(bothTrue)
                        
                        // Add reward to objective (-5 points per consecutive assignment)
                        softPenalties.add(
                            buildLinearExpr()
                                .add(LinearExpr.term(bothTrue, -5L))
                                .build()
                        )
                    }
                }
            }
        }
    }

    @SoftConstraint("Prefer larger bookings", weight = 2)
    fun preferLargerBookings() {
        // For each resource and date, reward larger bookings
        for (resourceIdx in 0L until problem.getResourceDomainSize()) {
            val resourceId = problem.getResourceIdByIndex(resourceIdx)
            
            for (dateIdx in 0L until problem.getDateDomainSize()) {
                val date = problem.getDateByIndex(dateIdx)
                
                // Group work units by activity
                val unitsByActivity = problem.getAllWorkUnitsIndexed().groupBy { it.activityId }
                
                for ((activityId, units) in unitsByActivity) {
                    // Count units assigned to this resource/date/activity
                    val assignmentVars = units.map { unit ->
                        val isAssigned = cpModel.newBoolVar("assigned_${unit.id}_${resourceId}_${date}")
                        
                        cpModel.addEquality(buildResourceExprLong(resourceVars[unit.id]!!, resourceIdx), 0L)
                            .onlyEnforceIf(isAssigned)
                        
                        cpModel.addEquality(buildDateExprLong(dateVars[unit.id]!!, dateIdx), 0L)
                            .onlyEnforceIf(isAssigned)
                        
                        isAssigned
                    }
                    
                    // Count total assignments
                    val count = LinearExpr.sum(assignmentVars.toTypedArray())
                    
                    // Add quadratic reward to objective (-2 * count^2)
                    // We use a piecewise linear function to approximate square
                    for (i in 1..8) {
                        val threshold = cpModel.newBoolVar("threshold_${i}_${activityId}_${resourceId}_${date}")
                        cpModel.addGreaterOrEqual(count, i.toLong()).onlyEnforceIf(threshold)
                        cpModel.addLessThan(count, i.toLong()).onlyEnforceIf(threshold.not())
                        
                        softPenalties.add(
                            buildLinearExpr()
                                .add(LinearExpr.term(threshold, -2L * i * i))  // Quadratic reward
                                .build()
                        )
                    }
                }
            }
        }
    }

    @SoftConstraint("Maximize margin", weight = 1)
    fun maximizeMargin() {
        for (unit in problem.getAllWorkUnitsIndexed()) {
            val activity = problem.getActivity(unit.activityId)!!
            if (activity.isCommercialRate) {
                // For each possible resource, calculate margin contribution
                for (resourceIdx in 0L until problem.getResourceDomainSize()) {
                    val resourceId = problem.getResourceIdByIndex(resourceIdx)
                    val resource = problem.getResource(resourceId)!!
                    val position = problem.getPosition(resource.positionId)!!
                    val margin = activity.calculateMargin(position.hourlyRate)
                    val marginScaled = Math.max(0L, (margin * 10).toLong())

                    if (marginScaled > 0) {
                        // Create indicator for this resource being selected
                        val isSelected = cpModel.newBoolVar("margin_res_${unit.id}_${resourceId}")
                        
                        cpModel.addEquality(buildResourceExprLong(resourceVars[unit.id]!!, resourceIdx), 0L)
                            .onlyEnforceIf(isSelected)
                        
                        // Add margin reward to objective
                        softPenalties.add(LinearExpr.term(isSelected, -marginScaled))
                    }
                }
            }
        }
    }

    @SoftConstraint("Prefer early start dates", weight = 3)
    fun preferEarlyStart() {
        // Group work units by activity
        val unitsByActivity = problem.getAllWorkUnitsIndexed()
            .groupBy { it.activityId }
        
        for ((activityId, units) in unitsByActivity) {
            // Create variable for average date of this activity
            val avgDate = cpModel.newIntVar(0, problem.getDateDomainSize() - 1,
                "avg_date_${activityId}")
            
            // Sum all dates
            val dateSum = LinearExpr.sum(units.map { dateVars[it.id]!! }.toTypedArray())
            
            // Average = sum / count (using integer division)
            cpModel.addEquality(dateSum, LinearExpr.term(avgDate, units.size.toLong()))
            
            // Add penalty based on average date
            softPenalties.add(
                buildLinearExpr()
                    .add(LinearExpr.term(avgDate, units.size.toLong()))  // Scale by activity size
                    .build()
            )
        }
    }

    // Helper method for linear expressions
    private fun CpModel.addLinearExpr(target: IntVar, coefficient: Long, indicator: Literal) {
        val expr = LinearExpr.newBuilder()
            .addTerm(indicator, coefficient)
            .build()
        addEquality(target, expr)
    }
}

// ============ BUILDER CLASS ============

class CpSatBuilder(
    private val model: SchedulingModel,
    private val timeLimitSeconds: Long = 30
) {

    fun solve(): ResourceScheduleSolution {
        val startTime = System.currentTimeMillis()

        // Initialize CP-SAT model
        model.cpModel = CpModel()

        // Create decision variables using domain sizes from problem container
        val resourceVarsMap = mutableMapOf<Long, IntVar>()
        val dateVarsMap = mutableMapOf<Long, IntVar>()
        for (unit in model.problem.getAllWorkUnitsIndexed()) {
            resourceVarsMap[unit.id] = model.cpModel.newIntVar(0L, (model.problem.getResourceDomainSize() - 1).toLong(), "resource_${unit.id}")
            dateVarsMap[unit.id] = model.cpModel.newIntVar(0L, (model.problem.getDateDomainSize() - 1).toLong(), "date_${unit.id}")
        }
        model.resourceVars = resourceVarsMap
        model.dateVars = dateVarsMap

        // Apply hard constraints
        val hardConstraintMethods = model::class.memberFunctions
            .filter { it.annotations.any { ann -> ann is HardConstraint } }

        for (method in hardConstraintMethods) {
            method.call(model)
        }

        // Apply soft constraints
        val softConstraintMethods = model::class.memberFunctions
            .filter { it.annotations.any { ann -> ann is SoftConstraint } }

        for (method in softConstraintMethods) {
            method.call(model)
        }

        // Set objective
        if (model.softPenalties.isNotEmpty()) {
            // Sum all penalties and rewards
            val objective = LinearExpr.sum(model.softPenalties.toTypedArray())
            model.cpModel.minimize(objective)
        }

        // Configure solver
        val solver = CpSolver()
        
        // Set solver parameters
        val parameters = solver.parameters
        parameters.maxTimeInSeconds = timeLimitSeconds.toDouble()
        parameters.numSearchWorkers = 8  // Use multiple threads
        parameters.logSearchProgress = true  // Enable logging
        
        // Set search strategies
        parameters.linearizationLevel = 2
        parameters.optimizeWithCore = true
        parameters.optimizeWithMaxHs = true
        parameters.useObjectiveLbSearch = true
        parameters.useLnsOnly = false

        // Solve
        val status = solver.solve(model.cpModel)
        val solveTime = System.currentTimeMillis() - startTime

        // Check status and extract solution
        when (status.toString()) {
            "OPTIMAL", "FEASIBLE" -> {
                val assignments = mutableMapOf<Long, Pair<ResourceID, LocalDate>>()
                for (unit in model.problem.getAllWorkUnitsIndexed()) {
                    val resourceIdx = solver.value(model.resourceVars[unit.id]!!)
                    val dateIdx = solver.value(model.dateVars[unit.id]!!)
                    val resourceId = model.problem.getResourceIdByIndex(resourceIdx)
                    val date = model.problem.getDateByIndex(dateIdx)
                    assignments[unit.id] = Pair(resourceId, date)
                }

                return ResourceScheduleSolution(
                    problem = model.problem,
                    assignments = assignments,
                    objectiveValue = solver.objectiveValue().toLong(),
                    solveTimeMs = solveTime
                )
            }
            "INFEASIBLE" -> throw RuntimeException("Problem is infeasible")
            "MODEL_INVALID" -> {
                throw RuntimeException("Model is invalid - check constraint implementation")
            }
            else -> throw RuntimeException("No solution found: $status")
        }
    }
}


## Load Data

Load the same test data from yaml 


In [9]:
// Custom deserializer for LocalDate
class LocalDateDeserializer : JsonDeserializer<LocalDate>() {
    override fun deserialize(p: JsonParser, ctxt: DeserializationContext): LocalDate {
        return LocalDate.parse(p.text)
    }
}

// Setup Jackson mapper
val mapper = ObjectMapper(YAMLFactory()).apply {
    registerModule(KotlinModule.Builder().build())
    registerModule(JavaTimeModule())
    val module = SimpleModule()
    module.addDeserializer(LocalDate::class.java, LocalDateDeserializer())
    registerModule(module)
}

// Data transfer classes for deserialization
data class DataFile(
    val departments: List<Department>,
    val positions: List<Position>,
    @JsonProperty("bankHolidaySets") val bankHolidaySets: List<BankHolidaySet>,
    val resources: List<Resource>,
    val projects: List<Project>,
    val activities: List<Activity>
)

// Load data
val dataFile = File("../data/data.yaml")
val data = mapper.readValue(dataFile, DataFile::class.java)

val departments = data.departments
val positions = data.positions
val bankHolidaySets = data.bankHolidaySets
val resources = data.resources
val projects = data.projects
val activities = data.activities

println("Loaded data:")
println("  Departments: ${departments.size}")
println("  Positions: ${positions.size}")
println("  Bank Holiday Sets: ${bankHolidaySets.size}")
println("  Resources: ${resources.size}")
println("  Projects: ${projects.size}")
println("  Activities: ${activities.size}")


Loaded data:
  Departments: 4
  Positions: 8
  Bank Holiday Sets: 2
  Resources: 10
  Projects: 4
  Activities: 10


In [10]:
// Function to get date range from projects
fun getProjectDateRange(projects: List<Project>): Pair<LocalDate, LocalDate> {
    // Get all non-null start and end dates
    val startDates = projects.mapNotNull { it.startDate }
    val endDates = projects.mapNotNull { it.endDate }
    
    if (startDates.isEmpty() || endDates.isEmpty()) {
        throw IllegalArgumentException("No valid project dates found")
    }
    
    // Find min start date and max end date
    val minStartDate = startDates.minOrNull()!!
    val maxEndDate = endDates.maxOrNull()!!
    
    return Pair(minStartDate, maxEndDate)
}

// Get date range from projects
val (minStartDate, maxEndDate) = getProjectDateRange(projects)
println("Project date range:")
println("  Start: $minStartDate")
println("  End: $maxEndDate")


Project date range:
  Start: 2025-09-22
  End: 2025-10-31


## Define Date Range

Create a date range for scheduling (September 29 to October 10).


In [11]:
// Generate date range starting from September 29th to end date
val startDate = LocalDate.of(2025, 9, 29)
val dateRange = mutableListOf<LocalDate>()
var currentDate = startDate
while (!currentDate.isAfter(maxEndDate)) {
    if (currentDate.dayOfWeek != DayOfWeek.SATURDAY &&
        currentDate.dayOfWeek != DayOfWeek.SUNDAY) {
        dateRange.add(currentDate)
    }
    currentDate = currentDate.plusDays(1)
}

println("Date range: ${dateRange.size} working days")
dateRange.forEach { println("  - $it (${it.dayOfWeek})") }


Date range: 25 working days
  - 2025-09-29 (MONDAY)
  - 2025-09-30 (TUESDAY)
  - 2025-10-01 (WEDNESDAY)
  - 2025-10-02 (THURSDAY)
  - 2025-10-03 (FRIDAY)
  - 2025-10-06 (MONDAY)
  - 2025-10-07 (TUESDAY)
  - 2025-10-08 (WEDNESDAY)
  - 2025-10-09 (THURSDAY)
  - 2025-10-10 (FRIDAY)
  - 2025-10-13 (MONDAY)
  - 2025-10-14 (TUESDAY)
  - 2025-10-15 (WEDNESDAY)
  - 2025-10-16 (THURSDAY)
  - 2025-10-17 (FRIDAY)
  - 2025-10-20 (MONDAY)
  - 2025-10-21 (TUESDAY)
  - 2025-10-22 (WEDNESDAY)
  - 2025-10-23 (THURSDAY)
  - 2025-10-24 (FRIDAY)
  - 2025-10-27 (MONDAY)
  - 2025-10-28 (TUESDAY)
  - 2025-10-29 (WEDNESDAY)
  - 2025-10-30 (THURSDAY)
  - 2025-10-31 (FRIDAY)


## Generate Work Units

Break each activity into 1-hour work units that will be assigned to resources and dates during solving.


In [12]:
// Generate work units from activities
var unitId = 1L
val workUnits = mutableListOf<WorkUnit>()

// Create one work unit for each hour of work needed
for (activity in activities) {
    repeat(activity.estimatedHours) {
        workUnits.add(
            WorkUnit(
                id = unitId++,
                activityId = activity.id,
                projectId = activity.projectId
            )
        )
    }
}

println("Created ${workUnits.size} work units (1 hour each) from ${activities.size} activities")
println("Activity breakdown:")
activities.forEach { activity ->
    println("  - ${activity.name}: ${activity.estimatedHours} hours")
}


Created 146 work units (1 hour each) from 10 activities
Activity breakdown:
  - Architecture Design: 13 hours
  - API Development: 18 hours
  - Integration Testing: 11 hours
  - Wireframing: 14 hours
  - Visual Design: 19 hours
  - Prototyping: 12 hours
  - Database Schema: 17 hours
  - Frontend Implementation: 22 hours
  - User Interviews: 13 hours
  - Data Analysis: 7 hours


## Create Problem and Solve


In [13]:
val problem = ResourceScheduleProblem(
    projects = projects,
    resources = resources,
    activities = activities,
    departments = departments,
    positions = positions,
    bankHolidaySets = bankHolidaySets,
    dateRange = dateRange,
    workUnits = workUnits
)

println("Problem created with:")
println("  - ${problem.resources.size} resources")
println("  - ${problem.activities.size} activities")
println("  - ${problem.projects.size} projects")
println("  - ${problem.workUnits.size} work units (hours) to schedule")
println("  - ${problem.dateRange.size} available dates")
println()

// Create the model and builder
val model = SchedulingModel(problem)
val builder = CpSatBuilder(model)

println("Solving...")
val solution = builder.solve()

println("\n=== SOLUTION ===")
println("Objective value: ${solution.objectiveValue}")
println("Solve time: ${solution.solveTimeMs}ms")
println("Assignments: ${solution.assignments.size}")


Problem created with:
  - 10 resources
  - 10 activities
  - 4 projects
  - 146 work units (hours) to schedule
  - 25 available dates

Solving...

=== SOLUTION ===
Objective value: -129227
Solve time: 34409ms
Assignments: 146


## Visualize Results
Display the schedule grouped by resource.


In [14]:
// Group assignments by resource and date
data class BookingGroup(val activityId: ActivityID, val resourceId: ResourceID, val date: LocalDate, val hours: Int)

val bookingsByResource = solution.assignments.entries
    .groupBy { it.value.first }  // Group by resourceId
    .mapValues { (_, entries) ->
        entries.groupBy { it.value.second }  // Group by date
            .mapValues { (_, dateEntries) ->
                dateEntries.groupBy { solution.problem.getWorkUnit(it.key)!!.activityId }
                    .map { (actId, actEntries) ->
                        BookingGroup(actId, entries.first().value.first, dateEntries.first().value.second, actEntries.size)
                    }
            }
    }

HTML(buildString {
    append("<div style='background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin-bottom: 20px;'>")
    append("<p style='font-size: x-large; margin: 5px 0;'><b>Objective:</b> ${solution.objectiveValue}</p>")
    append("<p style='font-size: large; margin: 5px 0;'><b>Solve time:</b> ${solution.solveTimeMs}ms</p>")
    append("<p style='margin: 5px 0;'><b>Resources:</b> ${problem.getResourceDomainSize()} | <b>Activities:</b> ${problem.getActivityDomainSize()} | <b>Work Units:</b> ${problem.getWorkUnitDomainSize()}</p>")
    append("</div>")

    append("<table style='border-collapse: collapse;'><tr style='background-color: #f0f0f0;'><th style='border: 1px solid #ddd; padding: 8px;'>Resource</th><th style='border: 1px solid #ddd; padding: 8px;'>Bookings</th></tr>")
    for (resource in problem.getAllResourcesIndexed()) {
        val resourceBookings = bookingsByResource[resource.id] ?: emptyMap()
        append("<tr><th style='border: 1px solid #ddd; padding: 8px; text-align: left;'>${resource.name} (${resource.positionId})</th>")
        append("<td style='border: 1px solid #ddd; padding: 8px;'>")

        if (resourceBookings.isNotEmpty()) {
            val sortedBookings = resourceBookings.entries.sortedBy { it.key }.flatMap { it.value }
            append(sortedBookings.map { booking ->
                val activity = problem.getActivity(booking.activityId)
                val project = problem.getProject(activity?.projectId ?: 0L)
                val position = problem.getPosition(resource.positionId)
                val margin = if (activity != null && position != null && activity.isCommercialRate) {
                    activity.calculateMargin(position.hourlyRate) * booking.hours
                } else 0.0
                "<b>${booking.date}</b> (${booking.date.dayOfWeek}): ${activity?.name} - ${project?.projectCode} - ${booking.hours}h (margin: ${'$'}${String.format("%.2f", margin)})"
            }.joinToString("<br/>"))
        } else {
            append("<i>No bookings</i>")
        }
        append("</td></tr>")
    }
    append("</table>")
})


Resource,Bookings
Zara Blackwood (JUNIOR_DEV),"2025-10-10 (FRIDAY): API Development - CS-2025 - 7h (margin: $770,00)2025-10-13 (MONDAY): API Development - CS-2025 - 1h (margin: $110,00)2025-10-14 (TUESDAY): API Development - CS-2025 - 1h (margin: $110,00)2025-10-16 (THURSDAY): Integration Testing - CS-2025 - 2h (margin: $180,00)"
Felix Moonstone (MID_DEV),"2025-09-29 (MONDAY): Architecture Design - CS-2025 - 4h (margin: $420,00)2025-09-30 (TUESDAY): Architecture Design - CS-2025 - 7h (margin: $735,00)2025-10-02 (THURSDAY): Architecture Design - CS-2025 - 2h (margin: $210,00)2025-10-20 (MONDAY): Database Schema - DAD-789 - 1h (margin: $100,00)2025-10-21 (TUESDAY): Database Schema - DAD-789 - 8h (margin: $800,00)2025-10-22 (WEDNESDAY): Database Schema - DAD-789 - 1h (margin: $100,00)2025-10-23 (THURSDAY): Frontend Implementation - DAD-789 - 6h (margin: $480,00)"
Nova Brightwell (SENIOR_DEV),"2025-10-21 (TUESDAY): Database Schema - DAD-789 - 7h (margin: $525,00)2025-10-23 (THURSDAY): Frontend Implementation - DAD-789 - 8h (margin: $440,00)"
River Ashford (JUNIOR_DESIGNER),"2025-09-29 (MONDAY): Wireframing - MAR-401 - 3h (margin: $285,00)2025-09-30 (TUESDAY): Wireframing - MAR-401 - 8h (margin: $760,00)2025-10-01 (WEDNESDAY): Wireframing - MAR-401 - 3h (margin: $285,00)2025-10-06 (MONDAY): User Interviews - URI-202 - 1h (margin: $65,00)2025-10-07 (TUESDAY): Prototyping - MAR-401 - 3h (margin: $315,00)2025-10-08 (WEDNESDAY): Prototyping - MAR-401 - 5h (margin: $525,00)2025-10-14 (TUESDAY): Prototyping - MAR-401 - 3h (margin: $315,00)2025-10-17 (FRIDAY): User Interviews - URI-202 - 7h (margin: $455,00)2025-10-20 (MONDAY): User Interviews - URI-202 - 1h (margin: $65,00)2025-10-21 (TUESDAY): User Interviews - URI-202 - 4h (margin: $260,00)2025-10-29 (WEDNESDAY): Prototyping - MAR-401 - 1h (margin: $105,00)"
Luna Silverstone (MID_DESIGNER),"2025-10-02 (THURSDAY): Visual Design - MAR-401 - 6h (margin: $540,00)2025-10-03 (FRIDAY): Visual Design - MAR-401 - 7h (margin: $630,00)2025-10-06 (MONDAY): Visual Design - MAR-401 - 6h (margin: $540,00)"
Atlas Thornfield (SENIOR_DESIGNER),No bookings
Sage Windermere (PM),No bookings
Phoenix Ravenscroft (BA),No bookings
Echo Silverstream (JUNIOR_DEV),"2025-10-03 (FRIDAY): API Development - CS-2025 - 1h (margin: $110,00)2025-10-07 (TUESDAY): API Development - CS-2025 - 2h (margin: $220,00)2025-10-08 (WEDNESDAY): API Development - CS-2025 - 4h (margin: $440,00)2025-10-09 (THURSDAY): API Development - CS-2025 - 2h (margin: $220,00)2025-10-15 (WEDNESDAY): Integration Testing - CS-2025 - 1h (margin: $90,00)2025-10-16 (THURSDAY): Integration Testing - CS-2025 - 7h (margin: $630,00)2025-10-17 (FRIDAY): Integration Testing - CS-2025 - 1h (margin: $90,00)2025-10-23 (THURSDAY): Frontend Implementation - DAD-789 - 8h (margin: $840,00)"
Aurora Nightshade (MID_DESIGNER),"2025-10-28 (TUESDAY): Data Analysis - URI-202 - 7h (margin: $350,00)"


## Resource View

In [15]:
// Group work units by resource and date, then by activity
data class ActivityBooking(val activityId: ActivityID, val projectId: ProjectID, val hours: Int)

val bookingsByResourceDate = solution.assignments.entries
    .groupBy { it.value.first }  // Group by resourceId
    .mapValues { (_, units) -> 
        units.groupBy { it.value.second }  // Group by date
            .mapValues { (_, dateUnits) ->
                // Group by activity within each date
                dateUnits.groupBy { solution.problem.getWorkUnit(it.key)!!.activityId }
                    .map { (actId, actUnits) ->
                        ActivityBooking(
                            actId, 
                            solution.problem.getWorkUnit(actUnits.first().key)!!.projectId,
                            actUnits.size
                        )
                    }
            }
    }

// Get unique dates sorted
val allDates = solution.problem.getAllDatesIndexed().sorted()

// Define colors for different projects using HSL color space
fun setHue(baseHex: String, hue: Float): String {
    val baseHSL = RGB(baseHex).toHSL()
    return HSL(hue, baseHSL.s, baseHSL.l).toSRGB().toHex()
}

val baseColor = "#FFB6C1"
// distribute colors evenly across the hue spectrum to avoid color clashes
val projectColorsList = (1..problem.getProjectDomainSize()).map { index -> 
    setHue(baseColor, 360f * index.toFloat() / problem.getProjectDomainSize()) 
}
val projectColors = problem.getAllProjectsIndexed().mapIndexed { index, project ->
    project.id to projectColorsList[index % projectColorsList.size]
}.toMap()

HTML(buildString {
    append("<style>")
    append(".gantt-table { border-collapse: collapse; font-size: 12px; width: 100%; }")
    append(".gantt-table th, .gantt-table td { border: 1px solid #ddd; padding: 8px; text-align: center; }")
    append(".gantt-table th { background-color: #f8f9fa; font-weight: bold; position: sticky; top: 0; }")
    append(".gantt-table td.resource-cell { text-align: left; background-color: #f8f9fa; font-weight: bold; min-width: 150px; }")
    append(".gantt-table td.date-cell { min-width: 100px; vertical-align: top; }")
    append(".booking-block { padding: 4px; margin: 2px; border-radius: 3px; font-size: 11px; }")
    append("</style>")
    
    append("<table class='gantt-table'>")
    
    // Header row with dates
    append("<tr>")
    append("<th>Resource</th>")
    append("<th>Department</th>")
    append("<th>Position</th>")
    for (date in allDates) {
        val dayOfWeek = date.dayOfWeek.toString().substring(0, 3)
        append("<th>")
        append("$dayOfWeek<br/>")
        append("${date.monthValue}/${date.dayOfMonth}")
        append("</th>")
    }
    append("</tr>")
    
    // Resource rows
    for (resource in problem.getAllResourcesIndexed()) {
        append("<tr>")
        append("<td class='resource-cell'>${resource.name}</td>")
        
        val department = problem.getDepartment(resource.departmentId)
        append("<td>${department?.name ?: resource.departmentId}</td>")
        
        val position = problem.getPosition(resource.positionId)
        append("<td>${position?.name ?: resource.positionId}</td>")
        
        // Date cells
        val resourceBookings = bookingsByResourceDate[resource.id] ?: emptyMap()
        for (date in allDates) {
            append("<td class='date-cell'>")
            
            val dayBookings = resourceBookings[date] ?: emptyList()
            var totalHours = 0
            
            for (booking in dayBookings) {
                val activity = problem.getActivity(booking.activityId)
                val project = problem.getProject(booking.projectId)
                val color = projectColors[project?.id] ?: "#E0E0E0"
                
                totalHours += booking.hours
                
                append("<div class='booking-block' style='background-color: $color;' title='${project?.name} - ${activity?.name} (${booking.hours}h)'>")
                append("<b>${activity?.code}</b><br/>")
                append("${booking.hours}h")
                append("</div>")
            }
            
            // Show total hours if there are bookings
            if (totalHours > 0) {
                append("<small><b>Total: ${totalHours}h</b></small>")
            }
            
            append("</td>")
        }
        
        append("</tr>")
    }
    
    append("</table>")
    
    // Legend
    append("<br/><div style='margin-top: 10px;'>")
    append("<b>Project Legend:</b><br/>")
    for (project in problem.getAllProjectsIndexed()) {
        val color = projectColors[project.id] ?: "#E0E0E0"
        append("<span style='display: inline-block; width: 15px; height: 15px; background-color: $color; border: 1px solid #999; margin-right: 5px;'></span>")
        append("${project.projectCode} - ${project.name}<br/>")
    }
    append("</div>")
})


Resource,Department,Position,MON9/29,TUE9/30,WED10/1,THU10/2,FRI10/3,MON10/6,TUE10/7,WED10/8,THU10/9,FRI10/10,MON10/13,TUE10/14,WED10/15,THU10/16,FRI10/17,MON10/20,TUE10/21,WED10/22,THU10/23,FRI10/24,MON10/27,TUE10/28,WED10/29,THU10/30,FRI10/31
Zara Blackwood,Engineering Department,Junior Developer,,,,,,,,,,CS-2025-0027hTotal: 7h,CS-2025-0021hTotal: 1h,CS-2025-0021hTotal: 1h,,CS-2025-0032hTotal: 2h,,,,,,,,,,,
Felix Moonstone,Engineering Department,Mid Developer,CS-2025-0014hTotal: 4h,CS-2025-0017hTotal: 7h,,CS-2025-0012hTotal: 2h,,,,,,,,,,,,DAD-789-0011hTotal: 1h,DAD-789-0018hTotal: 8h,DAD-789-0011hTotal: 1h,DAD-789-0026hTotal: 6h,,,,,,
Nova Brightwell,Engineering Department,Senior Developer,,,,,,,,,,,,,,,,,DAD-789-0017hTotal: 7h,,DAD-789-0028hTotal: 8h,,,,,,
River Ashford,Design Department,Junior Designer,MAR-401-0013hTotal: 3h,MAR-401-0018hTotal: 8h,MAR-401-0013hTotal: 3h,,,URI-202-0011hTotal: 1h,MAR-401-0033hTotal: 3h,MAR-401-0035hTotal: 5h,,,,MAR-401-0033hTotal: 3h,,,URI-202-0017hTotal: 7h,URI-202-0011hTotal: 1h,URI-202-0014hTotal: 4h,,,,,,MAR-401-0031hTotal: 1h,,
Luna Silverstone,Design Department,Mid Designer,,,,MAR-401-0026hTotal: 6h,MAR-401-0027hTotal: 7h,MAR-401-0026hTotal: 6h,,,,,,,,,,,,,,,,,,,
Atlas Thornfield,Design Department,Senior Designer,,,,,,,,,,,,,,,,,,,,,,,,,
Sage Windermere,Product Department,Product Manager,,,,,,,,,,,,,,,,,,,,,,,,,
Phoenix Ravenscroft,Operations Department,Business Analyst,,,,,,,,,,,,,,,,,,,,,,,,,
Echo Silverstream,Engineering Department,Junior Developer,,,,,CS-2025-0021hTotal: 1h,,CS-2025-0022hTotal: 2h,CS-2025-0024hTotal: 4h,CS-2025-0022hTotal: 2h,,,,CS-2025-0031hTotal: 1h,CS-2025-0037hTotal: 7h,CS-2025-0031hTotal: 1h,,,,DAD-789-0028hTotal: 8h,,,,,,
Aurora Nightshade,Design Department,Mid Designer,,,,,,,,,,,,,,,,,,,,,,URI-202-0027hTotal: 7h,,,


## Project View

In [16]:
// Group work units by project, activity, date, and resource
data class ResourceBooking(val resourceId: ResourceID, val hours: Int)

val bookingsByProjectActivity = solution.assignments.entries
    .groupBy { solution.problem.getWorkUnit(it.key)!!.projectId }
    .mapValues { (_, units) -> 
        units.groupBy { solution.problem.getWorkUnit(it.key)!!.activityId }
            .mapValues { (_, actUnits) ->
                actUnits.groupBy { it.value.second }  // Group by date
                    .mapValues { (_, dateUnits) ->
                        // Group by resource within each date
                        dateUnits.groupBy { it.value.first }
                            .map { (resId, resUnits) ->
                                ResourceBooking(resId, resUnits.size)
                            }
                    }
            }
    }

// Get unique dates sorted (reuse from earlier)
val projectGanttDates = solution.problem.getAllDatesIndexed().sorted()

// Define colors for different resources using HSL color space
val resourceColors = problem.getAllResourcesIndexed().mapIndexed { index, resource ->
    resource.id to projectColorsList[index % projectColorsList.size]
}.toMap()

HTML(buildString {
    append("<style>")
    append(".project-gantt-table { border-collapse: collapse; font-size: 12px; width: 100%; }")
    append(".project-gantt-table th, .project-gantt-table td { border: 1px solid #ddd; padding: 8px; text-align: center; }")
    append(".project-gantt-table th { background-color: #f8f9fa; font-weight: bold; }")
    append(".project-gantt-table td.project-cell { text-align: left; background-color: #e8f4f8; font-weight: bold; min-width: 150px; }")
    append(".project-gantt-table td.activity-cell { text-align: left; background-color: #f8f9fa; font-weight: bold; min-width: 200px; padding-left: 20px; }")
    append(".project-gantt-table td.date-cell { min-width: 100px; vertical-align: top; }")
    append(".resource-block { padding: 4px; margin: 2px; border-radius: 3px; font-size: 11px; }")
    append("</style>")
    
    append("<table class='project-gantt-table'>")
    
    // Header row with dates
    append("<tr>")
    append("<th>Project</th>")
    append("<th>Activity</th>")
    for (date in projectGanttDates) {
        val dayOfWeek = date.dayOfWeek.toString().substring(0, 3)
        append("<th>$dayOfWeek<br/>${date.monthValue}/${date.dayOfMonth}</th>")
    }
    append("</tr>")
    
    // Project and activity rows
    for (project in problem.getAllProjectsIndexed()) {
        val projectBookings = bookingsByProjectActivity[project.id] ?: emptyMap()
        val projectActivities = problem.getAllActivitiesIndexed()
            .filter { it.projectId == project.id }
            .sortedBy { it.order }
        
        if (projectActivities.isEmpty()) continue
        
        var firstActivityRow = true
        for (activity in projectActivities) {
            append("<tr>")
            
            if (firstActivityRow) {
                append("<td class='project-cell' rowspan='${projectActivities.size}'>")
                append("<b>${project.projectCode}</b><br/>${project.name}")
                append("</td>")
                firstActivityRow = false
            }
            
            append("<td class='activity-cell'>")
            append("<b>${activity.code}</b><br/>${activity.name}<br/>")
            append("<small>${activity.estimatedHours}h est.</small>")
            append("</td>")
            
            val activityBookings = projectBookings[activity.id] ?: emptyMap()
            for (date in projectGanttDates) {
                append("<td class='date-cell'>")
                
                val dayBookings = activityBookings[date] ?: emptyList()
                var totalHours = 0
                
                for (booking in dayBookings) {
                    val resource = problem.getResource(booking.resourceId)
                    val color = resourceColors[resource?.id] ?: "#E0E0E0"
                    totalHours += booking.hours
                    
                    val initials = resource?.name?.split(" ")?.map { it.first() }?.joinToString("") ?: "?"
                    append("<div class='resource-block' style='background-color: $color;'>")
                    append("<b>$initials</b><br/>${booking.hours}h")
                    append("</div>")
                }
                
                if (totalHours > 0) {
                    append("<small><b>Total: ${totalHours}h</b></small>")
                }
                
                append("</td>")
            }
            
            append("</tr>")
        }
    }
    
    append("</table>")
    
    append("<br/><div style='margin-top: 10px;'><b>Resource Legend:</b><br/>")
    for (resource in problem.getAllResourcesIndexed()) {
        val color = resourceColors[resource.id] ?: "#E0E0E0"
        val position = problem.getPosition(resource.positionId)
        val initials = resource.name.split(" ").map { it.first() }.joinToString("")
        append("<span style='display: inline-block; width: 15px; height: 15px; background-color: $color; border: 1px solid #999; margin-right: 5px;'></span>")
        append("<b>$initials</b> - ${resource.name} (${position?.name})<br/>")
    }
    append("</div>")
})


Project Activity MON 9/29 TUE 9/30 WED 10/1 THU 10/2 FRI 10/3 MON 10/6 TUE 10/7 WED 10/8 THU 10/9 FRI 10/10 MON 10/13 TUE 10/14 WED 10/15 THU 10/16 FRI 10/17 MON 10/20 TUE 10/21 WED 10/22 THU 10/23 FRI 10/24 MON 10/27 TUE 10/28 WED 10/29 THU 10/30 FRI 10/31 CS-2025 CloudSync Platform CS-2025-001 Architecture Design 13h est. FM 4h Total: 4h FM 7h Total: 7h FM 2h Total: 2h CS-2025-002 API Development 18h est. ES 1h Total: 1h ES 2h Total: 2h ES 4h Total: 4h ES 2h Total: 2h ZB 7h Total: 7h ZB 1h Total: 1h ZB 1h Total: 1h CS-2025-003 Integration Testing 11h est. ES 1h Total: 1h ES 7h ZB 2h Total: 9h ES 1h Total: 1h MAR-401 Mobile App Redesign MAR-401-001 Wireframing 14h est. RA 3h Total: 3h RA 8h Total: 8h RA 3h Total: 3h MAR-401-002 Visual Design 19h est. LS 6h Total: 6h LS 7h Total: 7h LS 6h Total: 6h MAR-401-003 Prototyping 12h est. RA 3h Total: 3h RA 5h Total: 5h RA 3h Total: 3h RA 1h Total: 1h DAD-789 Data Analytics Dashboard DAD-789-001 Database Schema 17h est. FM 1h Total: 1h NB 7h FM 8h Total: 15h FM 1h Total: 1h DAD-789-002 Frontend Implementation 22h est. FM 6h NB 8h ES 8h Total: 22h URI-202 User Research Initiative URI-202-001 User Interviews 13h est. RA 1h Total: 1h RA 7h Total: 7h RA 1h Total: 1h RA 4h Total: 4h URI-202-002 Data Analysis 7h est. AN 7h Total: 7h Resource Legend: ZB - Zara Blackwood (Junior Developer) FM - Felix Moonstone (Mid Developer) NB - Nova Brightwell (Senior Developer) RA - River Ashford (Junior Designer) LS - Luna Silverstone (Mid Designer) AT - Atlas Thornfield (Senior Designer) SW - Sage Windermere (Product Manager) PR - Phoenix Ravenscroft (Business Analyst) ES - Echo Silverstream (Junior Developer) AN - Aurora Nightshade (Mid Designer)

## Analyze Margin Optimization

Let's analyze the margin optimization results with a detailed breakdown.


In [17]:
var totalCommercialCost = 0.0
var totalResourceCost = 0.0

// Group work units by activity and resource for cleaner output
val unitsByActivityResource = solution.assignments.entries
    .groupBy { 
        val unit = solution.problem.getWorkUnit(it.key)!!
        Pair(unit.activityId, it.value.first)  // Group by (activityId, resourceId)
    }

data class MarginRow(
    val activityName: String,
    val resourceName: String,
    val hours: Int,
    val commercialRate: Double,
    val resourceRate: Double,
    val commercialCost: Double,
    val resourceCost: Double,
    val margin: Double,
    val marginPercent: Double
)

val marginRows = mutableListOf<MarginRow>()

for ((key, units) in unitsByActivityResource) {
    val (activityId, resourceId) = key
    val activity = problem.getActivity(activityId)
    val resource = problem.getResource(resourceId)
    val position = resource?.let { problem.getPosition(it.positionId) }
    
    if (activity != null && position != null) {
        val hours = units.size  // Each unit is 1 hour
        val commercialCost = activity.rate * hours
        val resourceCost = position.hourlyRate * hours
        val margin = if (activity.isCommercialRate) commercialCost - resourceCost else 0.0
        val marginPercent = if (commercialCost > 0) (margin / commercialCost) * 100 else 0.0
        
        totalCommercialCost += commercialCost
        totalResourceCost += resourceCost
        
        marginRows.add(MarginRow(
            activityName = activity.name,
            resourceName = resource.name,
            hours = hours,
            commercialRate = activity.rate,
            resourceRate = position.hourlyRate,
            commercialCost = commercialCost,
            resourceCost = resourceCost,
            margin = margin,
            marginPercent = marginPercent
        ))
    }
}

val totalMargin = totalCommercialCost - totalResourceCost
val totalMarginPercentage = if (totalCommercialCost > 0) (totalMargin / totalCommercialCost) * 100 else 0.0

HTML(buildString {
    append("<style>")
    append("@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');")
    append(".margin-container { font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; max-width: 100%; margin: 0 auto; }")
    append(".margin-header { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 32px; border-radius: 16px; margin-bottom: 24px; box-shadow: 0 10px 40px rgba(102, 126, 234, 0.3); }")
    append(".margin-header h2 { margin: 0 0 16px 0; color: white; font-size: 28px; font-weight: 700; letter-spacing: -0.5px; }")
    append(".margin-stats { display: flex; gap: 32px; flex-wrap: wrap; }")
    append(".margin-stat { flex: 1; min-width: 200px; }")
    append(".margin-stat-label { color: rgba(255,255,255,0.8); font-size: 12px; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 500; margin-bottom: 4px; }")
    append(".margin-stat-value { color: white; font-size: 32px; font-weight: 700; line-height: 1; }")
    append(".margin-stat-subvalue { color: rgba(255,255,255,0.9); font-size: 18px; font-weight: 600; margin-top: 4px; }")
    append(".margin-table-container { background: white; border-radius: 16px; overflow: hidden; box-shadow: 0 4px 24px rgba(0,0,0,0.06); }")
    append(".margin-table { border-collapse: separate; border-spacing: 0; font-size: 14px; width: 100%; }")
    append(".margin-table thead { background: #f8fafc; }")
    append(".margin-table th { padding: 16px 20px; text-align: left; font-weight: 600; font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.5px; border-bottom: 2px solid #e2e8f0; }")
    append(".margin-table th.right { text-align: right; }")
    append(".margin-table td { padding: 20px; border-bottom: 1px solid #f1f5f9; color: #334155; }")
    append(".margin-table tbody tr { transition: all 0.2s ease; }")
    append(".margin-table tbody tr:hover { background-color: #f8fafc; transform: scale(1.001); }")
    append(".margin-table tbody tr:last-child td { border-bottom: none; }")
    append(".margin-table .activity-name { font-weight: 600; color: #0f172a; font-size: 15px; }")
    append(".margin-table .resource-name { color: #64748b; font-size: 14px; }")
    append(".margin-table .numeric { text-align: right; font-variant-numeric: tabular-nums; font-weight: 500; }")
    append(".margin-table .positive { color: #10b981; font-weight: 600; }")
    append(".margin-table .badge { display: inline-block; padding: 4px 10px; border-radius: 6px; font-size: 13px; font-weight: 600; }")
    append(".margin-table .badge-high { background: #dcfce7; color: #15803d; }")
    append(".margin-table .totals-row { background: linear-gradient(to right, #f8fafc, #f1f5f9) !important; border-top: 2px solid #e2e8f0 !important; }")
    append(".margin-table .totals-row td { padding: 24px 20px; font-weight: 700; font-size: 15px; color: #0f172a; border-bottom: none !important; }")
    append(".margin-table .totals-row .positive { color: #059669; font-size: 18px; }")
    append("</style>")
    
    append("<div class='margin-container'>")
    append("<div class='margin-header'>")
    append("<h2>💰 Margin Analysis</h2>")
    append("<div class='margin-stats'>")
    append("<div class='margin-stat'>")
    append("<div class='margin-stat-label'>Total Margin</div>")
    append("<div class='margin-stat-value'>${'$'}${String.format("%,.0f", totalMargin)}</div>")
    append("<div class='margin-stat-subvalue'>${String.format("%.1f", totalMarginPercentage)}% margin rate</div>")
    append("</div>")
    append("<div class='margin-stat'>")
    append("<div class='margin-stat-label'>Revenue</div>")
    append("<div class='margin-stat-value'>${'$'}${String.format("%,.0f", totalCommercialCost)}</div>")
    append("</div>")
    append("<div class='margin-stat'>")
    append("<div class='margin-stat-label'>Activities</div>")
    append("<div class='margin-stat-value'>${marginRows.size}</div>")
    append("</div>")
    append("</div>")
    append("</div>")
    
    append("<div class='margin-table-container'>")
    append("<table class='margin-table'>")
    append("<thead><tr>")
    append("<th>Activity</th>")
    append("<th>Resource</th>")
    append("<th class='right'>Hours</th>")
    append("<th class='right'>Commercial Rate</th>")
    append("<th class='right'>Resource Rate</th>")
    append("<th class='right'>Commercial Cost</th>")
    append("<th class='right'>Resource Cost</th>")
    append("<th class='right'>Margin</th>")
    append("<th class='right'>Margin %</th>")
    append("</tr></thead><tbody>")
    
    // Sort by margin descending
    for (row in marginRows.sortedByDescending { it.margin }) {
        append("<tr>")
        append("<td><div class='activity-name'>${row.activityName}</div></td>")
        append("<td><div class='resource-name'>${row.resourceName}</div></td>")
        append("<td class='numeric'>${row.hours}</td>")
        append("<td class='numeric'>${'$'}${String.format("%.0f", row.commercialRate)}</td>")
        append("<td class='numeric'>${'$'}${String.format("%.0f", row.resourceRate)}</td>")
        append("<td class='numeric'>${'$'}${String.format("%,.0f", row.commercialCost)}</td>")
        append("<td class='numeric'>${'$'}${String.format("%,.0f", row.resourceCost)}</td>")
        append("<td class='numeric positive'>${'$'}${String.format("%,.0f", row.margin)}</td>")
        append("<td class='numeric'><span class='badge badge-high'>${String.format("%.0f", row.marginPercent)}%</span></td>")
        append("</tr>")
    }
    
    // Totals row
    val totalHours = marginRows.sumOf { it.hours }
    append("<tr class='totals-row'>")
    append("<td colspan='2'>TOTALS</td>")
    append("<td class='numeric'>${totalHours}</td>")
    append("<td colspan='2'></td>")
    append("<td class='numeric'>${'$'}${String.format("%,.0f", totalCommercialCost)}</td>")
    append("<td class='numeric'>${'$'}${String.format("%,.0f", totalResourceCost)}</td>")
    append("<td class='numeric positive'>${'$'}${String.format("%,.0f", totalMargin)}</td>")
    append("<td class='numeric positive'>${String.format("%.0f", totalMarginPercentage)}%</td>")
    append("</tr>")
    
    append("</tbody></table>")
    append("</div>")
    append("</div>")
})


Activity,Resource,Hours,Commercial Rate,Resource Rate,Commercial Cost,Resource Cost,Margin,Margin %
Visual Design,Luna Silverstone,19,$170,$80,$3 230,$1 520,$1 710,53%
Architecture Design,Felix Moonstone,13,$180,$75,$2 340,$975,$1 365,58%
Wireframing,River Ashford,14,$150,$55,$2 100,$770,$1 330,63%
Prototyping,River Ashford,12,$160,$55,$1 920,$660,$1 260,66%
Database Schema,Felix Moonstone,10,$175,$75,$1 750,$750,$1 000,57%
API Development,Zara Blackwood,9,$160,$50,$1 440,$450,$990,69%
API Development,Echo Silverstream,9,$160,$50,$1 440,$450,$990,69%
User Interviews,River Ashford,13,$120,$55,$1 560,$715,$845,54%
Frontend Implementation,Echo Silverstream,8,$155,$50,$1 240,$400,$840,68%
Integration Testing,Echo Silverstream,9,$140,$50,$1 260,$450,$810,64%
